# Netflix Content Strategy Analytics — Extended ETL

This notebook extends the existing Netflix ETL by creating analytical features required for the Content Strategy Dashboard.

In [ ]:
!pip install -q pyspark==3.5.3


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("Netflix-Content-Strategy-ETL")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

spark


In [ ]:
CSV_PATH = "/content/netflix_titles.csv"

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
)

print("Rows:", raw.count())
print("Columns:", len(raw.columns))
raw.printSchema()


In [ ]:
# Basic quality checks
null_counts = raw.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in raw.columns
])

null_counts.show(truncate=False)


In [ ]:
clean = (
    raw
    .dropna(subset=["show_id", "title"])
    .dropDuplicates(["show_id"])
    .na.fill({
        "country": "Unknown",
        "director": "Unknown",
        "cast": "Unknown",
        "rating": "Not Rated",
        "listed_in": "Unknown"
    })
    .withColumn("release_year", F.col("release_year").cast("int"))
    .withColumn(
        "date_added",
        F.to_date(F.trim(F.col("date_added")), "MMMM d, yyyy")
    )
    .withColumn(
        "content_age",
        F.year(F.current_date()) - F.col("release_year")
    )
    .withColumn(
        "primary_country",
        F.trim(F.split(F.col("country"), ",").getItem(0))
    )
    .withColumn(
        "genre",
        F.explode(F.split(F.col("listed_in"), r",\s*"))
    )
)

print("Rows after genre explosion:", clean.count())


In [ ]:
# Extended analytical features

extended = (
    clean
    .withColumn(
        "content_age_group",
        F.when(F.col("content_age") <= 2, "Very Recent")
         .when(F.col("content_age") <= 5, "Recent")
         .when(F.col("content_age") <= 10, "Mature")
         .when(F.col("content_age") <= 20, "Older")
         .otherwise("Legacy")
    )
    .withColumn(
        "release_decade",
        (F.floor(F.col("release_year") / 10) * 10).cast("int")
    )
    .withColumn(
        "is_recent",
        F.when(F.col("content_age") <= 5, 1).otherwise(0)
    )
)

extended.select(
    "show_id", "title", "type", "release_year",
    "content_age", "content_age_group",
    "release_decade", "is_recent",
    "primary_country", "genre"
).show(10, truncate=False)


In [ ]:
# Keep the same streaming-era focus as the existing dashboard
dashboard = (
    extended
    .filter(F.col("release_year").isNotNull())
    .filter(F.col("release_year") >= 2008)
)

print("Dashboard rows:", dashboard.count())


In [ ]:
# Save the extended dataset
OUTPUT_PATH = "/content/content_strategy_extended.csv"

dashboard.toPandas().to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)


In [ ]:
# Optional: inspect title-level KPI grain
title_level = (
    dashboard
    .groupBy("show_id")
    .agg(
        F.first("title").alias("title"),
        F.first("type").alias("type"),
        F.first("primary_country").alias("primary_country"),
        F.first("release_year").alias("release_year"),
        F.first("rating").alias("rating"),
        F.first("content_age").alias("content_age"),
        F.first("content_age_group").alias("content_age_group"),
        F.first("release_decade").alias("release_decade"),
        F.first("is_recent").alias("is_recent")
    )
)

title_level.show(5, truncate=False)
print("Distinct titles:", title_level.count())


## Output

Use `content_strategy_extended.csv` as the input to the dashboard-preparation workflow or load the transformed data into MySQL.